# N117 · DP空间与转移优化

**目标：** 先证明依赖与候选支配再删除状态或转移。

**先修：** N098, N056, N031, N099。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 滚动数组；前缀和；单调队列；窗口DP；保留恢复路径的代价。

## 从问题到算法

DP优化先写明候选集合，再寻找支配关系。窗口最大转移用单调队列丢弃“更旧且值不更大”的状态，因为它们更早过期且收益不占优。约束子序列允许重新开始，跳跃游戏必须从位置0到末尾，转移中不能随意加入0。逆序对计数的转移是连续区间求和，可用滑动和。

## 最小实现

**本章接口：** `constrained_subset_sum(nums, k)`、`max_result(nums, k)`、`k_inverse_pairs(n, k)`

In [1]:
from collections import deque
MOD=10**9+7

def constrained_subset_sum(nums,k):
    if not nums or k<1: raise ValueError('nonempty nums and positive k required')
    q=deque(); best=float('-inf')
    for i,x in enumerate(nums):
        while q and q[0][0]<i-k: q.popleft()
        value=x+max(0,q[0][1] if q else 0); best=max(best,value)
        while q and q[-1][1]<=value: q.pop()
        q.append((i,value))
    return best

def max_result(nums,k):
    if not nums or k<1: raise ValueError('nonempty nums and positive k required')
    q=deque([(0,nums[0])]); value=nums[0]
    for i in range(1,len(nums)):
        while q[0][0]<i-k: q.popleft()
        value=nums[i]+q[0][1]
        while q and q[-1][1]<=value: q.pop()
        q.append((i,value))
    return value

def k_inverse_pairs(n,k):
    if min(n,k)<0 or k>n*(n-1)//2: return 0
    dp=[1]+[0]*k
    for length in range(1,n+1):
        nxt=[0]*(k+1); window=0
        for inv in range(k+1):
            window+=dp[inv]
            if inv>=length: window-=dp[inv-length]
            nxt[inv]=window%MOD
        dp=nxt
    return dp[k]

## 正确性、前提与复杂度

队列按下标递增、值严格递减；过期项从头删除，被新状态支配的项从尾删除，每项至多进出一次。将最大元素n插入n−1个数的排列中，会新增0..n−1个逆序对，因此dp[n][k]是旧行一段连续和，滑动更新保持这个区间不变量。

**代价：** 两个窗口DP O(n)时间O(min(n,k))辅助空间；逆序对O(nk)时间O(k)空间。计数模10^9+7。队列优化不自动保留最优路径；恢复路径需要另存前驱O(n)。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

nums=[1,-1,-2,4,-7,3]; k=2; q=deque([(0,nums[0])]); rows=[(0,nums[0],list(q))]
for i in range(1,len(nums)):
    while q[0][0]<i-k: q.popleft()
    value=nums[i]+q[0][1]
    while q and q[-1][1]<=value: q.pop()
    q.append((i,value)); rows.append((i,value,list(q)))
show_table(['位置','最优到达分数','单调候选(位置,值)'],rows)

位置,最优到达分数,"单调候选(位置,值)"
0,1,"[(0, 1)]"
1,0,"[(0, 1), (1, 0)]"
2,-1,"[(0, 1), (1, 0), (2, -1)]"
3,4,"[(3, 4)]"
4,-3,"[(3, 4), (4, -3)]"
5,7,"[(5, 7)]"


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert max_result([1,-1,-2,4,-7,3],2)==7
assert constrained_subset_sum([-3,-1,-2],2)==-1
from itertools import product,permutations
for a in product((-2,0,3),repeat=6):
    for k in (1,2,4):
        dp=[a[0]]; sub=[]
        for i,x in enumerate(a):
            if i: dp.append(x+max(dp[max(0,i-k):i]))
            sub.append(x+max([0]+sub[max(0,i-k):i]))
        assert max_result(a,k)==dp[-1]
        assert constrained_subset_sum(a,k)==max(sub)
for n in range(8):
    counts={}
    for p in permutations(range(n)):
        inv=sum(p[i]>p[j] for i in range(n) for j in range(i+1,n)); counts[inv]=counts.get(inv,0)+1
    for k in range(n*(n-1)//2+2): assert k_inverse_pairs(n,k)==counts.get(k,0)
print('N117: 所有本章断言通过')

N117: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 分别优化时间和空间并报告改变。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 解释滚动数组后如何恢复路径。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 1425. Constrained Subsequence Sum（canonical）
- 1696. Jump Game VI（canonical）
- 629. K Inverse Pairs Array（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。